# Laya in PyTorch: check the browser demo's numbers

This notebook runs the original **Laya English** model (`convaiinnovations/laya`, Apache-2.0) in PyTorch, so you can compare its answers with the ONNX build in [layaForWeb](https://github.com/vishalmysore/layaForWeb).

**You do not need a GPU.** The model has 421M parameters and each call takes a few seconds on a CPU. A GPU runtime only makes it faster. Use the default runtime (Runtime > Change runtime type > CPU).

The first run downloads the 843 MB checkpoint from Hugging Face. No token is needed.

In [ ]:
!pip -q install laya

In [ ]:
import torch, laya

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

agent = laya.load("convaiinnovations/laya", device=device)
print("parameters:", sum(p.numel() for p in agent.model.parameters()))
print("dtype:", next(agent.model.parameters()).dtype)

## 1. The agent-guardrail case from the article

Same text and same three questions as the "Agent guardrail" example in the demo page.

In [ ]:
state = ("Agent plan: run `DELETE FROM customers WHERE last_login < '2020-01-01'` "
         "on the production database. No backup has been taken and no human has reviewed this command.")

questions = {
    "safe_without_approval": {"type": "noul", "instructions": "Is this safe to run without a human approving it first?"},
    "destructive":           {"type": "noul", "instructions": "The command is destructive and cannot be undone"},
    "needs_human":           {"type": "noul", "instructions": "A human should approve this command before it runs"},
}

result = agent.system_one(state, questions)

# What the browser demo (q8e8 build, WASM) showed for P(yes) on the same input.
browser = {"safe_without_approval": 0.828, "destructive": 0.649, "needs_human": 0.327}

print(f"{'question':24} {'PyTorch':>8} {'browser':>8} {'diff':>7}   confidence")
for name, a in result["answers"].items():
    p = a["noul"]
    print(f"{name:24} {p:8.3f} {browser[name]:8.3f} {abs(p - browser[name]):7.3f}   {a['confidence']:.3f}")

On a CPU this printed 0.833 / 0.632 / 0.337 for P(yes). A GPU can differ in the third decimal place because of different floating-point kernels.

## 2. Your own data

Edit `state` and `questions`. Three question types:

- `choice`: `criteria` is a dict of `label: description`; you get one probability per label.
- `score`: `criteria` is an ordered list of levels; you get a probability per level and a continuous score.
- `noul`: no criteria; you get P(yes) for the statement in `instructions`.

In [ ]:
import json

state = {
    "ticket": {
        "subject": "App crashes on launch",
        "text": "Since the last update, the app closes as soon as I open it. I have a demo in one hour!",
    }
}

questions = {
    "team":    {"type": "choice", "instructions": "Which team should handle this?",
                "criteria": {"bug": "Something is broken", "how_to": "A usage question", "sales": "Pricing or plans"}},
    "urgency": {"type": "score",  "instructions": "How urgent is this?",
                "criteria": ["Can wait", "This week", "Today", "Right now"]},
    "angry":   {"type": "noul",   "instructions": "The customer sounds angry"},
}

print(json.dumps(agent.system_one(state, questions)["answers"], indent=2))

## 3. Many texts at once

To see how often the model is confident and how often it would hand off to a person, loop over your own examples and apply a threshold.

In [ ]:
texts = [
    "Please cancel my subscription, I no longer need it.",
    "Where can I find the invoice for last month?",
    "Your app deleted all my data and I want to speak to a manager NOW.",
]
q = {"intent": {"type": "choice", "instructions": "What does the customer want?",
                "criteria": {"cancel": "Cancel the service", "billing": "Billing question", "complaint": "Complaint or escalation"}}}

THRESHOLD = 0.90
for t in texts:
    a = agent.system_one(t, q)["answers"]["intent"]
    action = "auto" if a["confidence"] >= THRESHOLD else "human"
    print(f"{a['choice']:10} conf={a['confidence']:.3f} -> {action:5}  {t[:60]}")